In [1]:
import gc
import getpass
import hashlib
import importlib.metadata
import json
from pathlib import Path
import glob
import os
import platform
import re
import shutil
import subprocess
import uuid

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from proyecto_mlops_equipo_9.config import (
    PROJ_ROOT,
    CLEAN_DATA_DIR,
    PROCESSED_DATA_DIR 
)
from proyecto_mlops_equipo_9.dataset import (
    COLUMNAS,
)

2026-10-25 17:09:31.494 | INFO     | proyecto_mlops_equipo_9.config:<module>:11 - PROJ_ROOT path is: C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9


In [2]:
print(PROJ_ROOT)
print(CLEAN_DATA_DIR)
print(PROCESSED_DATA_DIR)

C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9
C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9\data\silver\version_001\viajes
C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9\data\processed


In [27]:
output_path: Path = PROCESSED_DATA_DIR / "datos_feature_engineered.parquet"
print(output_path)

C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9\data\processed\datos_feature_engineered.parquet


In [4]:
files = glob.glob(os.path.join(CLEAN_DATA_DIR, "*.parquet"))
files

['C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\data\\silver\\version_001\\viajes\\viajes_2025_01.parquet',
 'C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\data\\silver\\version_001\\viajes\\viajes_2025_02.parquet',
 'C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\data\\silver\\version_001\\viajes\\viajes_2025_03.parquet',
 'C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\data\\silver\\version_001\\viajes\\viajes_2025_04.parquet',
 'C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\data\\silver\\version_001\\viajes\\viajes_2025_05.parquet',
 'C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\data\\silver\\version_001\\viajes\\viajes_2025_06.parquet',
 'C:\\Users\\paqui\\OneDrive\\Documentos\\Maestria\\MLOPS_Equipo\\proyecto_mlops_equipo_9\\dat

In [5]:
df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

print(df.head())
print(df.shape)

   viaje_id  usuario_id genero  anio_nacimiento              inicio  \
0  37162342      601273      M           1982.0 2025-01-01 00:00:44   
1  37162343     1571524      M           2003.0 2025-01-01 00:04:11   
2  37162344     1435200      F           1984.0 2025-01-01 00:05:27   
3  37162345      126327      M           1988.0 2025-01-01 00:07:47   
4  37162346      416121      M           1994.0 2025-01-01 00:08:52   

                  fin  origen_id  destino_id  duracion_min  fila_csv  ...  \
0 2025-01-01 00:11:29        211         395     10.750000         2  ...   
1 2025-01-01 00:15:07         35         260     10.933333         3  ...   
2 2025-01-01 00:15:23         35         260      9.933333         4  ...   
3 2025-01-01 00:17:42        190          11      9.916667         5  ...   
4 2025-01-01 00:18:10        273           8      9.300000         6  ...   

                           updt_dt  version_datos  version_limpieza  \
0 2026-10-24 04:39:09.381909+00:00    v

In [6]:
df['dataset_id'].unique()

<ArrowStringArray>
['7813aa70c6be348833409a731db19b72dae6001eaca9b1a20a5f0a82dc76c7a3']
Length: 1, dtype: str

In [7]:
df.head()

,viaje_id,usuario_id,genero,anio_nacimiento,inicio,fin,origen_id,destino_id,duracion_min,fila_csv,...,updt_dt,version_datos,version_limpieza,dataset_id,lote_id,catalogo_version,catalogo_sha256,codigo_limpieza_sha256,archivo_origen,sha256_origen
0,37162342,601273,M,1982.0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395,10.750000,2,...,2026-10-24 04:39:09.381909+00:00,version_001,1.0.0,7813aa70c6be348833409a731db19b72dae6001eaca9b1...,6e500b52-8b09-40c2-abe3-5fde1e7d6ab2,nomenclatura_2026_09.csv,206e45fe633922b634d858142fa21e6fd9f4c421b6435c...,ea5f19e6aec5829d24d41411126def82eb826b7e6e7f20...,data/raw/viajes/datos_abiertos_2025_01.csv,90ee75701a69374926af0400d204f3280baed0e3c1a1f9...
1,37162343,1571524,M,2003.0,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260,10.933333,3,...,2026-10-24 04:39:09.381909+00:00,version_001,1.0.0,7813aa70c6be348833409a731db19b72dae6001eaca9b1...,6e500b52-8b09-40c2-abe3-5fde1e7d6ab2,nomenclatura_2026_09.csv,206e45fe633922b634d858142fa21e6fd9f4c421b6435c...,ea5f19e6aec5829d24d41411126def82eb826b7e6e7f20...,data/raw/viajes/datos_abiertos_2025_01.csv,90ee75701a69374926af0400d204f3280baed0e3c1a1f9...
2,37162344,1435200,F,1984.0,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260,9.933333,4,...,2026-10-24 04:39:09.381909+00:00,version_001,1.0.0,7813aa70c6be348833409a731db19b72dae6001eaca9b1...,6e500b52-8b09-40c2-abe3-5fde1e7d6ab2,nomenclatura_2026_09.csv,206e45fe633922b634d858142fa21e6fd9f4c421b6435c...,ea5f19e6aec5829d24d41411126def82eb826b7e6e7f20...,data/raw/viajes/datos_abiertos_2025_01.csv,90ee75701a69374926af0400d204f3280baed0e3c1a1f9...
3,37162345,126327,M,1988.0,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11,9.916667,5,...,2026-10-24 04:39:09.381909+00:00,version_001,1.0.0,7813aa70c6be348833409a731db19b72dae6001eaca9b1...,6e500b52-8b09-40c2-abe3-5fde1e7d6ab2,nomenclatura_2026_09.csv,206e45fe633922b634d858142fa21e6fd9f4c421b6435c...,ea5f19e6aec5829d24d41411126def82eb826b7e6e7f20...,data/raw/viajes/datos_abiertos_2025_01.csv,90ee75701a69374926af0400d204f3280baed0e3c1a1f9...
4,37162346,416121,M,1994.0,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8,9.300000,6,...,2026-10-24 04:39:09.381909+00:00,version_001,1.0.0,7813aa70c6be348833409a731db19b72dae6001eaca9b1...,6e500b52-8b09-40c2-abe3-5fde1e7d6ab2,nomenclatura_2026_09.csv,206e45fe633922b634d858142fa21e6fd9f4c421b6435c...,ea5f19e6aec5829d24d41411126def82eb826b7e6e7f20...,data/raw/viajes/datos_abiertos_2025_01.csv,90ee75701a69374926af0400d204f3280baed0e3c1a1f9...


In [8]:
df.columns

Index(['viaje_id', 'usuario_id', 'genero', 'anio_nacimiento', 'inicio', 'fin',
       'origen_id', 'destino_id', 'duracion_min', 'fila_csv', 'mes_archivo',
       'flag_duplicado_exacto', 'flag_id_conflictivo',
       'flag_id_entre_archivos', 'flag_id_esencial_nulo',
       'flag_fecha_invalida', 'flag_fuera_mes', 'flag_duracion_no_positiva',
       'viaje_corto', 'corto_distinta_estacion', 'flag_misma_estacion_corta',
       'duracion_larga', 'flag_estacion_sin_catalogo', 'flag_sin_coordenadas',
       'velocidad_kmh', 'flag_velocidad_imposible', 'anio_nacimiento_original',
       'flag_edad_imposible', 'flag_nacimiento_no_numerico', 'edad',
       'genero_original', 'flag_genero_invalido', 'motivo_exclusion',
       'es_valido', 'usuario', 'insrt_dt', 'updt_dt', 'version_datos',
       'version_limpieza', 'dataset_id', 'lote_id', 'catalogo_version',
       'catalogo_sha256', 'codigo_limpieza_sha256', 'archivo_origen',
       'sha256_origen'],
      dtype='str')

In [9]:
df[['viaje_id', 'usuario_id', 'genero', 'anio_nacimiento', 'inicio', 'fin',
       'origen_id', 'destino_id', 'duracion_min', 'fila_csv', 'mes_archivo',
       'flag_duplicado_exacto', 'flag_id_conflictivo',
       'flag_id_entre_archivos', 'flag_id_esencial_nulo',
       'flag_fecha_invalida', 'flag_fuera_mes', 'flag_duracion_no_positiva',
       'viaje_corto', 'corto_distinta_estacion', 'flag_misma_estacion_corta',
       'duracion_larga', 'flag_estacion_sin_catalogo', 'flag_sin_coordenadas',
       'velocidad_kmh', 'flag_velocidad_imposible', 'anio_nacimiento_original',
       'flag_edad_imposible', 'flag_nacimiento_no_numerico', 'edad',
       'genero_original', 'flag_genero_invalido', 'motivo_exclusion',
       'es_valido']]

,viaje_id,usuario_id,genero,anio_nacimiento,inicio,fin,origen_id,destino_id,duracion_min,fila_csv,...,velocidad_kmh,flag_velocidad_imposible,anio_nacimiento_original,flag_edad_imposible,flag_nacimiento_no_numerico,edad,genero_original,flag_genero_invalido,motivo_exclusion,es_valido
0,37162342,601273,M,1982.0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395,10.750000,2,...,7.556001,False,1982.0,False,False,43.0,M,False,<NA>,True
1,37162343,1571524,M,2003.0,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260,10.933333,3,...,2.954598,False,2003.0,False,False,22.0,M,False,<NA>,True
2,37162344,1435200,F,1984.0,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260,9.933333,4,...,3.252041,False,1984.0,False,False,41.0,F,False,<NA>,True
3,37162345,126327,M,1988.0,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11,9.916667,5,...,14.062676,False,1988.0,False,False,37.0,M,False,<NA>,True
4,37162346,416121,M,1994.0,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8,9.300000,6,...,9.910337,False,1994.0,False,False,31.0,M,False,<NA>,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7644404,46255553,4630316,M,1999.0,2026-09-30 23:56:38,2026-10-01 00:14:52,73,73,18.233333,490052,...,0.000000,False,1999.0,False,False,27.0,M,False,<NA>,True
7644405,46255554,116275,F,1990.0,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44,2.383333,490053,...,5.313680,False,1990.0,False,False,36.0,F,False,<NA>,True
7644406,46255556,4621160,M,1970.0,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,5.066667,490054,...,14.133330,False,1970.0,False,False,56.0,M,False,<NA>,True
7644407,46255557,4560372,M,1987.0,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271,6.733333,490056,...,15.058860,False,1987.0,False,False,39.0,M,False,<NA>,True


# Decisión de columnas

* Debido a que se considerará que cada renglón es un caso independiente no es necesario conservar **viaje_id**.
* Tambien para no realizar un análisis con datos específicos de usuarios se tomo la decisión de tampoco utilizar **usuario_id**, **genero**, **anio_nacimiento**, **edad**, **genero_original**, **anio_nacimiento_original**.
* Todas las columnas de control tambien se van a eliminar **mes_archivo**, **fila_csv**, **motivo_exclusion**,**es_valido**, **usuario**, **insrt_dt**, **updt_dt**, **version_datos**, **version_limpieza**, **dataset_id**, **lote_id**, **catalogo_version**, **catalogo_sha256**, **codigo_limpieza_sha256**, **archivo_origen**, **sha256_origen**
* Columnas utilizadas para detectar casos invalidos en el proceso de limpieza son eliminadas, essto debido a que son falsas. **flag_duplicado_exacto**, **flag_id_conflictivo**, **flag_id_entre_archivos**, **flag_id_esencial_nulo**, **flag_fecha_invalida**, **flag_fuera_mes**, **flag_duracion_no_positiva****flag_nacimiento_no_numerico**, **flag_genero_invalido**, **flag_velocidad_imposible**, **flag_misma_estacion_corta**, **duracion_larga**, **flag_estacion_sin_catalogo**, **flag_sin_coordenadas**, **flag_velocidad_imposible**, **flag_edad_imposible**


In [10]:
columnas_feature_ini = ['inicio', 'fin',
       'origen_id', 'destino_id', 'duracion_min',
       'viaje_corto', 'corto_distinta_estacion',
       'velocidad_kmh']

In [11]:
df[columnas_feature_ini].nunique()

inicio                     6818966
fin                        6831476
origen_id                      470
destino_id                     470
duracion_min                  6729
viaje_corto                      2
corto_distinta_estacion          2
velocidad_kmh              4385218
dtype: int64

In [12]:
df[columnas_feature_ini]

,inicio,fin,origen_id,destino_id,duracion_min,viaje_corto,corto_distinta_estacion,velocidad_kmh
0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395,10.750000,False,False,7.556001
1,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260,10.933333,False,False,2.954598
2,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260,9.933333,False,False,3.252041
3,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11,9.916667,False,False,14.062676
4,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8,9.300000,False,False,9.910337
...,...,...,...,...,...,...,...,...
7644404,2026-09-30 23:56:38,2026-10-01 00:14:52,73,73,18.233333,False,False,0.000000
7644405,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44,2.383333,False,False,5.313680
7644406,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,5.066667,False,False,14.133330
7644407,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271,6.733333,False,False,15.058860


In [13]:
df_ini = df[columnas_feature_ini].copy()
df_ini

,inicio,fin,origen_id,destino_id,duracion_min,viaje_corto,corto_distinta_estacion,velocidad_kmh
0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395,10.750000,False,False,7.556001
1,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260,10.933333,False,False,2.954598
2,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260,9.933333,False,False,3.252041
3,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11,9.916667,False,False,14.062676
4,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8,9.300000,False,False,9.910337
...,...,...,...,...,...,...,...,...
7644404,2026-09-30 23:56:38,2026-10-01 00:14:52,73,73,18.233333,False,False,0.000000
7644405,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44,2.383333,False,False,5.313680
7644406,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,5.066667,False,False,14.133330
7644407,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271,6.733333,False,False,15.058860


In [14]:
#Se convierten los valores a tipo de dato de fecha y tiempo
df_ini["inicio"] = pd.to_datetime(df_ini["inicio"])
df_ini["fin"] = pd.to_datetime(df_ini["fin"])
df_ini.head()

,inicio,fin,origen_id,destino_id,duracion_min,viaje_corto,corto_distinta_estacion,velocidad_kmh
0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395,10.750000,False,False,7.556001
1,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260,10.933333,False,False,2.954598
2,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260,9.933333,False,False,3.252041
3,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11,9.916667,False,False,14.062676
4,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8,9.300000,False,False,9.910337


In [15]:
salidas = (
    df_ini.groupby([
        pd.Grouper(key="inicio", freq="15min"),
        "origen_id"
    ])
    .agg(
        salidas=("origen_id", "size"),
        duracion_prom_salidas=("duracion_min", "mean"),
        velocidad_prom_salidas=("velocidad_kmh", "mean"),
        proporcion_viajes_cortos_salidas=("viaje_corto", "mean"),
        proporcion_cortos_distinta_estacion_salidas=(
            "corto_distinta_estacion", "mean"
        )
    )
    .reset_index()
    .rename(columns={
        "inicio": "inicio_intervalo",
        "origen_id": "area_id"
    })
)
salidas

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas
0,2025-01-01 00:00:00,21,1,5.333333,10.271100,0.0,0.0
1,2025-01-01 00:00:00,34,1,2.116667,11.145827,0.0,0.0
2,2025-01-01 00:00:00,35,2,10.433333,3.103320,0.0,0.0
3,2025-01-01 00:00:00,54,2,9.008333,3.919751,0.0,0.0
4,2025-01-01 00:00:00,156,1,13.350000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...
4891705,2026-09-30 23:45:00,341,2,9.475000,14.404231,0.0,0.0
4891706,2026-09-30 23:45:00,354,1,4.100000,13.307365,0.0,0.0
4891707,2026-09-30 23:45:00,364,1,8.300000,12.544805,0.0,0.0
4891708,2026-09-30 23:45:00,428,1,4.100000,18.613838,0.0,0.0


In [16]:
llegadas = (
    df_ini.groupby([
        pd.Grouper(key="fin", freq="15min"),
        "destino_id"
    ])
    .agg(
        llegadas=("origen_id", "size"),
        duracion_prom_llegadas=("duracion_min", "mean"),
        velocidad_prom_llegadas=("velocidad_kmh", "mean"),
        proporcion_viajes_cortos_llegadas=("viaje_corto", "mean"),
        proporcion_cortos_distinta_estacion_llegadas=(
            "corto_distinta_estacion", "mean"
        )
    )
    .reset_index()
    .rename(columns={
        "fin": "inicio_intervalo",
        "destino_id": "area_id"
    })
)
llegadas

,inicio_intervalo,area_id,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,proporcion_viajes_cortos_llegadas,proporcion_cortos_distinta_estacion_llegadas
0,2025-01-01 00:00:00,8,1,2.116667,11.145827,0.0,0.0
1,2025-01-01 00:00:00,54,1,2.400000,0.000000,0.0,0.0
2,2025-01-01 00:00:00,395,1,10.750000,7.556001,0.0,0.0
3,2025-01-01 00:15:00,8,2,9.133333,10.094544,0.0,0.0
4,2025-01-01 00:15:00,11,1,9.916667,14.062676,0.0,0.0
...,...,...,...,...,...,...,...
4883353,2026-10-01 00:00:00,324,1,17.700000,10.649865,0.0,0.0
4883354,2026-10-01 00:00:00,350,2,13.958333,7.272215,0.0,0.0
4883355,2026-10-01 00:00:00,413,1,5.066667,14.133330,0.0,0.0
4883356,2026-10-01 00:00:00,428,1,22.400000,7.212091,0.0,0.0


In [17]:
agregados = pd.merge(
    salidas,
    llegadas,
    on=["inicio_intervalo", "area_id"],
    how="outer"
)
agregados

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,proporcion_viajes_cortos_llegadas,proporcion_cortos_distinta_estacion_llegadas
0,2025-01-01,8,NaN,NaN,NaN,NaN,NaN,1.0,2.116667,11.145827,0.0,0.0
1,2025-01-01,21,1.0,5.333333,10.271100,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,2025-01-01,34,1.0,2.116667,11.145827,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,2025-01-01,35,2.0,10.433333,3.103320,0.0,0.0,NaN,NaN,NaN,NaN,NaN
4,2025-01-01,54,2.0,9.008333,3.919751,0.0,0.0,1.0,2.400000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-10-01,324,NaN,NaN,NaN,NaN,NaN,1.0,17.700000,10.649865,0.0,0.0
7328184,2026-10-01,350,NaN,NaN,NaN,NaN,NaN,2.0,13.958333,7.272215,0.0,0.0
7328185,2026-10-01,413,NaN,NaN,NaN,NaN,NaN,1.0,5.066667,14.133330,0.0,0.0
7328186,2026-10-01,428,NaN,NaN,NaN,NaN,NaN,1.0,22.400000,7.212091,0.0,0.0


In [18]:
agregados = pd.merge(
    salidas,
    llegadas,
    on=["inicio_intervalo", "area_id"],
    how="outer"
)

agregados[["salidas", "llegadas"]] = (
    agregados[["salidas", "llegadas"]]
    .fillna(0)
    .astype(int)
)

agregados = agregados.sort_values(
    ["area_id","inicio_intervalo"]
).reset_index(drop=True)
agregados

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,proporcion_viajes_cortos_llegadas,proporcion_cortos_distinta_estacion_llegadas
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,0.0,0.0
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,0.0,0.0
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,NaN,NaN
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,NaN,NaN
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,0.0,0.0
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,0.0,0.0
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,0.0,0.0
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,0.0,0.0


In [19]:
for col in ["salidas", "llegadas"]:
    agregados[f"{col}_siguiente_hora"] = (
        agregados.groupby("area_id")[col]
        .transform(
            lambda s: s.shift(-1)
            .rolling(4, min_periods=4)
            .sum()
        )
    )
agregados

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,proporcion_viajes_cortos_llegadas,proporcion_cortos_distinta_estacion_llegadas,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,0.0,0.0,NaN,NaN
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,0.0,0.0,NaN,NaN
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,NaN,NaN,2.0,2.0
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,0.0,0.0,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,0.0,0.0,0.0,4.0
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,0.0,0.0,1.0,4.0
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,0.0,0.0,2.0,4.0
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,0.0,0.0,2.0,4.0


In [20]:
agregados['salidas_siguiente_hora'].isnull().mean()*100

np.float64(0.025654363670801025)

In [21]:
agregados

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,proporcion_viajes_cortos_llegadas,proporcion_cortos_distinta_estacion_llegadas,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,0.0,0.0,NaN,NaN
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,0.0,0.0,NaN,NaN
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,NaN,NaN,2.0,2.0
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,0.0,0.0,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,0.0,0.0,0.0,4.0
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,0.0,0.0,1.0,4.0
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,0.0,0.0,2.0,4.0
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,0.0,0.0,2.0,4.0


In [22]:
# Asegurar que la columna de fecha tenga formato datetime
agregados["inicio_intervalo"] = pd.to_datetime(
    agregados["inicio_intervalo"]
)

# Características temporales básicas
agregados["hora"] = agregados["inicio_intervalo"].dt.hour
agregados["dia_semana"] = agregados["inicio_intervalo"].dt.dayofweek
agregados["dia_mes"] = agregados["inicio_intervalo"].dt.day
agregados["mes"] = agregados["inicio_intervalo"].dt.month
agregados["dia_anio"] = agregados["inicio_intervalo"].dt.dayofyear
agregados["anio"] = agregados["inicio_intervalo"].dt.year

# Variable binaria: fin de semana
agregados["es_fin_semana"] = (
    agregados["dia_semana"] >= 5
).astype(int)

# Hora pico (ajustable según los patrones de tu ciudad)
agregados["es_hora_pico"] = (
    agregados["hora"].between(7, 9)
    | agregados["hora"].between(16, 19)
).astype(int)

# Codificación cíclica de la hora (ciclo de 24 horas)
hora_decimal = (
    agregados["inicio_intervalo"].dt.hour
    + agregados["inicio_intervalo"].dt.minute / 60
)

agregados["hora_sin"] = np.sin(
    2 * np.pi * hora_decimal / 24
)
agregados["hora_cos"] = np.cos(
    2 * np.pi * hora_decimal / 24
)

# Codificación cíclica del día de la semana (ciclo de 7 días)
agregados["dia_semana_sin"] = np.sin(
    2 * np.pi * agregados["dia_semana"] / 7
)
agregados["dia_semana_cos"] = np.cos(
    2 * np.pi * agregados["dia_semana"] / 7
)

agregados.head()

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_mes,mes,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,...,1,1,1,2025,0,0,1.000000,6.123234e-17,0.974928,-0.222521
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,...,1,1,1,2025,0,0,0.997859,-6.540313e-02,0.974928,-0.222521
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,...,1,1,1,2025,0,1,0.793353,-6.087614e-01,0.974928,-0.222521
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,...,1,1,1,2025,0,1,0.555570,-8.314696e-01,0.974928,-0.222521
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,...,1,1,1,2025,0,0,0.382683,-9.238795e-01,0.974928,-0.222521


# Llenar valores vacíos de llegadas y salidas

In [23]:
agregados["salidas"] = agregados["salidas"].fillna(0)
agregados["llegadas"] = agregados["llegadas"].fillna(0)
agregados["salidas_siguiente_hora"] = agregados["salidas_siguiente_hora"].fillna(0)
agregados["llegadas_siguiente_hora"] = agregados["llegadas_siguiente_hora"].fillna(0)

In [24]:
agregados

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_mes,mes,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,...,1,1,1,2025,0,0,1.000000,6.123234e-17,0.974928,-0.222521
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,...,1,1,1,2025,0,0,0.997859,-6.540313e-02,0.974928,-0.222521
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,...,1,1,1,2025,0,1,0.793353,-6.087614e-01,0.974928,-0.222521
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,...,1,1,1,2025,0,1,0.555570,-8.314696e-01,0.974928,-0.222521
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,...,1,1,1,2025,0,0,0.382683,-9.238795e-01,0.974928,-0.222521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,...,30,9,273,2026,0,1,-0.980785,-1.950903e-01,0.974928,-0.222521
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,...,30,9,273,2026,0,1,-0.997859,6.540313e-02,0.974928,-0.222521
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,...,30,9,273,2026,0,1,-0.965926,2.588190e-01,0.974928,-0.222521
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,...,30,9,273,2026,0,1,-0.946930,3.214395e-01,0.974928,-0.222521


In [25]:
objetivos = ["salidas_siguiente_hora", "llegadas_siguiente_hora"]

agregados = agregados[
    [col for col in agregados.columns if col not in objetivos]
    + objetivos
]

In [26]:
agregados

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,...,1,2025,0,0,1.000000,6.123234e-17,0.974928,-0.222521,0.0,0.0
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,...,1,2025,0,0,0.997859,-6.540313e-02,0.974928,-0.222521,0.0,0.0
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,...,1,2025,0,1,0.793353,-6.087614e-01,0.974928,-0.222521,0.0,0.0
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,...,1,2025,0,1,0.555570,-8.314696e-01,0.974928,-0.222521,2.0,2.0
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,...,1,2025,0,0,0.382683,-9.238795e-01,0.974928,-0.222521,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,...,273,2026,0,1,-0.980785,-1.950903e-01,0.974928,-0.222521,0.0,4.0
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,...,273,2026,0,1,-0.997859,6.540313e-02,0.974928,-0.222521,1.0,4.0
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,...,273,2026,0,1,-0.965926,2.588190e-01,0.974928,-0.222521,2.0,4.0
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,...,273,2026,0,1,-0.946930,3.214395e-01,0.974928,-0.222521,2.0,4.0


# Verificar salida del script utilizando mlflow

In [28]:
output_df = pd.read_parquet(output_path)
output_df

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,...,1,2025,0,0,1.000000,6.123234e-17,0.974928,-0.222521,0.0,0.0
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,...,1,2025,0,0,0.997859,-6.540313e-02,0.974928,-0.222521,0.0,0.0
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,...,1,2025,0,1,0.793353,-6.087614e-01,0.974928,-0.222521,0.0,0.0
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,...,1,2025,0,1,0.555570,-8.314696e-01,0.974928,-0.222521,2.0,2.0
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,...,1,2025,0,0,0.382683,-9.238795e-01,0.974928,-0.222521,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,...,273,2026,0,1,-0.980785,-1.950903e-01,0.974928,-0.222521,0.0,4.0
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,...,273,2026,0,1,-0.997859,6.540313e-02,0.974928,-0.222521,1.0,4.0
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,...,273,2026,0,1,-0.965926,2.588190e-01,0.974928,-0.222521,2.0,4.0
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,...,273,2026,0,1,-0.946930,3.214395e-01,0.974928,-0.222521,2.0,4.0
